# [drosophila_body_orientation_predictor](https://github.com/nehalsinghmangat/drosophila_body_orientation_predictor) — ANN trained on high-confidence headings

The ANN of `02_training_and_testing_the_ann.ipynb`, trained only on frames whose heading correction is confident: `heading_confidence > CONF_THRESHOLD` (0.99). `heading_confidence` is the forward-backward posterior of the front/back sign chosen by `heading_correction_viterbi` (notebook 01); on the hand-labelled trajectories in `heading_labeling/`, the Viterbi error falls from ~27 % of frames at confidence < 0.7 to 1.5 % at ≥ 0.999. Leaving the uncertain frames out removes most 180° label errors from the training set, at the cost of fewer frames, and the frames it removes are not spread evenly over flight regimes (see below).

Everything else is as in notebook 02, so that the two models differ only in their training frames:

- the same trajectory-grouped train/test split (seed 42, 33 % test), so both models are scored on the same held-out trajectories;
- the architecture selected there (3 hidden layers × 30 ReLU units, Table 6; the search is not repeated here), the same four inputs, wind-rotation augmentation, geodesic loss with per-trajectory-length sample weights, Adam, 150 epochs, batch 256, and the same inner validation split (seed 123);
- the same five seeds (42, 7, 13, 99, 123); seed 42 is the canonical model.

Both models are then evaluated on the held-out trajectories twice: on all their frames, and only on their frames with `heading_confidence > CONF_THRESHOLD`, where the ground truth itself is most trustworthy. The per-frame errors are written to `../tables/`; `07_full_vs_high_confidence_comparison.ipynb` plots the comparison.

**Training vs. loading.** `RETRAIN = True` trains the five models (a few minutes) and writes them to `../models/ann_model_conf0.99.keras` (seed 42) and `../models/ensemble_conf0.99/ann_seed_<seed>.keras`; `RETRAIN = False` (the default) loads them. The full-data models are always loaded from `../models/ensemble/`, as trained by notebook 02.

## Setup

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# TensorFlow must load before pandas: pandas loads pyarrow, whose bundled abseil then takes over TensorFlow's
# mutex code and model.fit / model.predict deadlock (the kernel sits at 0% CPU).
import keras
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, os.path.abspath('../utils'))
from utils import (circular_distance, pull_out_individual_trajectories, augment_trajectories, create_model,
                   geodesic_loss, trajectory_length_sample_weights)

RANDOM_SEED = 42
SEEDS = [42, 7, 13, 99, 123]
CONF_THRESHOLD = 0.99                # training (and the second evaluation) uses frames with heading_confidence > this
ARCH_LAYERS, ARCH_NEURONS = 3, 30    # selected by the architecture search in 02_training_and_testing_the_ann.ipynb
# True: train the five high-confidence models. False: load them from ../models/.
RETRAIN = False

MODEL_PATH = f'../models/ann_model_conf{CONF_THRESHOLD}.keras'
ENSEMBLE_DIR = f'../models/ensemble_conf{CONF_THRESHOLD}'
FULL_ENSEMBLE_DIR = '../models/ensemble'
RESULTS = '../tables/'
os.makedirs(RESULTS, exist_ok=True)
REGIMES = ['upwind', 'crosswind', 'downwind']


def classify_direction(angle_rad):
    # Wind blows toward +x, so upwind = -x: heading within 45 deg of 0 is downwind, within 45 deg of 180 is upwind.
    deg = np.degrees(np.arctan2(np.sin(angle_rad), np.cos(angle_rad)))
    return np.where(np.abs(deg) <= 45, 'downwind', np.where(np.abs(np.abs(deg) - 180) <= 45, 'upwind', 'crosswind'))


final_all_fly_data = pd.read_csv('../trainingdata/04_trainingdata/all_wind_heading_and_trajectories_augmented_corrected_filtered_headx_heady.csv')
print(f"{len(final_all_fly_data)} frames, {final_all_fly_data['trajec_objid'].nunique()} trajectories")

# same trajectory-grouped train/test split as 02_training_and_testing_the_ann.ipynb
traj_ids = final_all_fly_data['trajec_objid'].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(np.zeros((len(traj_ids), 1)), groups=traj_ids))
train_ids = set(traj_ids[train_idx])
test_ids = set(traj_ids[test_idx])
print(f"Train: {len(train_ids)} trajectories   Test: {len(test_ids)} trajectories")

112083 frames, 3321 trajectories
Train: 2225 trajectories   Test: 1096 trajectories


## The high-confidence training set

Trajectories are separated first (`pull_out_individual_trajectories` splits the data where the timestamp returns to 0, so frames cannot be removed before this step), then each training trajectory keeps only its frames with `heading_confidence > CONF_THRESHOLD`. A trajectory left with no frames is dropped. The model sees one frame at a time (no time-delay embedding), so gaps inside a trajectory do not matter; the sample weights (`1/len(trajectory)`) are computed over the frames each trajectory keeps, so every training trajectory still contributes equally to the loss.

In [2]:
fly_traj_list = pull_out_individual_trajectories(final_all_fly_data, min_traj_length=12)
train_trajectories = [t for t in fly_traj_list if t['trajec_objid'].iloc[0] in train_ids]
test_trajectories = [t for t in fly_traj_list if t['trajec_objid'].iloc[0] in test_ids]

hc_train_trajectories = [t[t['heading_confidence'] > CONF_THRESHOLD] for t in train_trajectories]
hc_train_trajectories = [t for t in hc_train_trajectories if len(t)]
n_all = sum(len(t) for t in train_trajectories)
n_hc = sum(len(t) for t in hc_train_trajectories)
print(f'training trajectories: {len(train_trajectories)} -> {len(hc_train_trajectories)} with at least one frame above {CONF_THRESHOLD}')
print(f'training frames:       {n_all} -> {n_hc} ({100 * n_hc / n_all:.1f} %)')

train_frames = pd.concat(train_trajectories, ignore_index=True)
kept = (train_frames.assign(regime=classify_direction(train_frames['heading_angle'].values),
                            kept=train_frames['heading_confidence'] > CONF_THRESHOLD)
        .groupby('regime')['kept'].agg(frames='size', kept='sum').reindex(REGIMES))
kept['% kept'] = (100 * kept['kept'] / kept['frames']).round(1)
display(kept)

ann_input_names = ['groundspeed', 'groundspeed_angle', 'airspeed', 'airspeed_angle']
n_input = len(ann_input_names)
n_output = 2
KEEP_COLS = ['heading_angle_x', 'heading_angle_y', 'trajec_objid']

hc_train_original = augment_trajectories(hc_train_trajectories, input_names=ann_input_names, output_names=KEEP_COLS)
hc_train_wind = augment_trajectories(hc_train_trajectories, input_names=ann_input_names, output_names=KEEP_COLS, wind_augment=True, wrap_angles=True)
hc_train_all = pd.concat([hc_train_original, hc_train_wind], ignore_index=True)
print(f"Train rows (original + rotated): {len(hc_train_all):,}")

Detected 3321 trajectories.
3321 trajectories with more than 12 timesteps.
training trajectories: 2225 -> 1938 with at least one frame above 0.99
training frames:       75141 -> 61925 (82.4 %)


,frames,kept,% kept
regime,,,
upwind,50501,46225,91.5
crosswind,18244,12841,70.4
downwind,6396,2859,44.7


Train rows (original + rotated): 123,850


## Training

In [3]:
def grouped_inner_val_split_with_weights(X, y, groups, val_fraction=0.2, seed=123):
    unique_groups = groups.unique()
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    dummy = np.zeros((len(unique_groups), 1))
    subtrain_gidx, val_gidx = next(gss2.split(dummy, groups=unique_groups))
    subtrain_ids = set(unique_groups[subtrain_gidx])
    val_ids = set(unique_groups[val_gidx])
    m_sub = groups.isin(subtrain_ids)
    m_val = groups.isin(val_ids)
    sw_sub = trajectory_length_sample_weights(groups.values)[m_sub.values]
    return (X[m_sub].values, y[m_sub].values, sw_sub), (X[m_val].values, y[m_val].values)


def load_saved_model(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found; set RETRAIN = True to train it")
    return keras.models.load_model(path, compile=False)


X_train = hc_train_all.iloc[:, 0:n_input].reset_index(drop=True)
y_train = hc_train_all[['heading_angle_x', 'heading_angle_y']].reset_index(drop=True)
groups_train = hc_train_all['trajec_objid'].reset_index(drop=True)

hc_models, full_models = {}, {}
for seed in SEEDS:
    path = f'{ENSEMBLE_DIR}/ann_seed_{seed}.keras'
    if RETRAIN:
        keras.utils.set_random_seed(seed)
        (Xtr, ytr, sw_tr), (Xval, yval) = grouped_inner_val_split_with_weights(X_train, y_train, groups_train, seed=123)
        m = create_model(n_input=n_input, n_output=n_output, neurons=ARCH_NEURONS, layers=ARCH_LAYERS, loss=geodesic_loss)
        m.fit(Xtr, ytr, sample_weight=sw_tr, validation_data=(Xval, yval), epochs=150, batch_size=256, verbose=0)
        os.makedirs(ENSEMBLE_DIR, exist_ok=True)
        m.save(path)
        if seed == RANDOM_SEED:
            m.save(MODEL_PATH)
        print(f'seed={seed}: trained on {len(Xtr):,} rows (validation {len(Xval):,}), saved {path}')
    hc_models[seed] = load_saved_model(MODEL_PATH if seed == RANDOM_SEED else path)
    full_models[seed] = load_saved_model(f'{FULL_ENSEMBLE_DIR}/ann_seed_{seed}.keras')
print(f'loaded {len(hc_models)} high-confidence and {len(full_models)} full-data models')

loaded 5 high-confidence and 5 full-data models


## Evaluation on the held-out trajectories

Every frame of the held-out trajectories (no confidence filter) is predicted by the five full-data and the five high-confidence models. The errors are scored as in notebook 02 (median per-trajectory angular error, overall and per regime, regime from the true heading), on two sets of frames:

- **all frames** of the held-out trajectories, as in notebook 02;
- **confident frames**: only the held-out frames with `heading_confidence > CONF_THRESHOLD`. Held-out trajectories with no such frame drop out of this set, and since confidence is lowest off the upwind axis, this set is weighted further toward upwind flight.

In [4]:
test_embedded = augment_trajectories(test_trajectories, input_names=ann_input_names, output_names=KEEP_COLS)
X_test = test_embedded.iloc[:, 0:n_input].values.astype('float32')
# frame metadata in the same order (augment_trajectories rounds its output, so take the confidence from the source)
frames = pd.concat([t.reset_index(drop=True) for t in test_trajectories], ignore_index=True)
assert (frames['trajec_objid'].values == test_embedded['trajec_objid'].values).all()

test_frames = pd.DataFrame({
    'trajec_objid': frames['trajec_objid'].values,
    'frame': frames.groupby('trajec_objid').cumcount().values,
    'windspeed': frames['windspeed'].values,
    'heading_confidence': frames['heading_confidence'].values,
    'regime': classify_direction(frames['heading_angle'].values),
    'heading_angle': frames['heading_angle'].values,
})
true_angle = np.arctan2(test_embedded['heading_angle_y'].values, test_embedded['heading_angle_x'].values)
for name, models in [('full', full_models), ('conf', hc_models)]:
    for seed, m in models.items():
        p = m(X_test, training=False).numpy()   # model.predict can deadlock when called many times
        test_frames[f'err_{name}_s{seed}'] = np.degrees(circular_distance(true_angle, np.arctan2(p[:, 1], p[:, 0])))

FRAME_ERRORS = f'{RESULTS}ann_conf{CONF_THRESHOLD}_test_frame_errors.csv'
test_frames.to_csv(FRAME_ERRORS, index=False)
confident = test_frames['heading_confidence'] > CONF_THRESHOLD
print(f'{len(test_frames)} held-out frames in {test_frames["trajec_objid"].nunique()} trajectories; '
      f'{confident.sum()} confident frames ({100 * confident.mean():.1f} %) in {test_frames.loc[confident, "trajec_objid"].nunique()} trajectories')
print('saved', FRAME_ERRORS)

36942 held-out frames in 1096 trajectories; 30170 confident frames (81.7 %) in 957 trajectories
saved ../tables/ann_conf0.99_test_frame_errors.csv


In [5]:
def bootstrap_ci(values, n_boot=5000, ci=95, seed=0):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boots = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_medians = np.median(boots, axis=1)
    lo, hi = np.percentile(boot_medians, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(np.median(values)), float(lo), float(hi)


EVAL_SETS = {'all frames': np.ones(len(test_frames), bool), f'confidence > {CONF_THRESHOLD}': confident.values}
MODEL_LABELS = {'full': 'all training frames', 'conf': f'training frames with confidence > {CONF_THRESHOLD}'}

rows = []
for eval_set, mask in EVAL_SETS.items():
    f = test_frames[mask]
    for name in MODEL_LABELS:
        for seed in SEEDS:
            col = f'err_{name}_s{seed}'
            groups = {'overall': f.groupby('trajec_objid')[col].median()}
            per_reg = f.groupby(['regime', 'trajec_objid'])[col].median()
            groups.update({r: per_reg.loc[r] for r in REGIMES})
            for regime, v in groups.items():
                med, lo, hi = bootstrap_ci(v.values)
                rows.append({'eval_set': eval_set, 'model': name, 'seed': seed, 'regime': regime,
                             'n_trajectories': len(v), 'median': med, 'ci95_lo': lo, 'ci95_hi': hi})
summary = pd.DataFrame(rows)
SUMMARY = f'{RESULTS}ann_conf{CONF_THRESHOLD}_comparison_summary.csv'
summary.to_csv(SUMMARY, index=False)
print('saved', SUMMARY)

# canonical (seed 42) models, with 95% CI
s42 = summary[summary['seed'] == RANDOM_SEED].copy()
s42['median [95% CI]'] = s42.apply(lambda r: f"{r['median']:.2f} [{r['ci95_lo']:.2f}, {r['ci95_hi']:.2f}]", axis=1)
table = s42.pivot_table(index=['eval_set', 'regime'], columns='model', values='median [95% CI]', aggfunc='first')
table['n'] = s42[s42['model'] == 'full'].set_index(['eval_set', 'regime'])['n_trajectories']
table = table.reindex([(e, r) for e in EVAL_SETS for r in ['overall'] + REGIMES])[['n', 'full', 'conf']]
table.columns = ['n trajectories', 'full-data model', 'high-confidence model']
print('Median per-trajectory error (deg), seed-42 models:')
display(table)

# spread over the five seeds: the seed-42 models above are one draw each, so compare the seed distributions too
spread = (summary.groupby(['eval_set', 'regime', 'model'])['median']
          .agg(lambda v: f'{v.mean():.2f} ± {v.std():.2f} ({v.min():.2f}–{v.max():.2f})')
          .unstack('model').reindex([(e, r) for e in EVAL_SETS for r in ['overall'] + REGIMES])[['full', 'conf']])
spread.columns = ['full-data model', 'high-confidence model']
print('Median per-trajectory error (deg) over the five seeds: mean ± sd (min–max)')
display(spread)

saved ../tables/ann_conf0.99_comparison_summary.csv
Median per-trajectory error (deg), seed-42 models:


n trajectories       full-data model  \
eval_set          regime                                            
all frames        overall              1096     4.96 [4.65, 5.44]   
                  upwind                903     5.08 [4.80, 5.70]   
                  crosswind             614  20.16 [16.79, 23.86]   
                  downwind              242     5.64 [4.31, 7.27]   
confidence > 0.99 overall               957     4.46 [4.13, 4.79]   
                  upwind                821     4.59 [4.19, 4.88]   
                  crosswind             478   11.94 [9.94, 15.44]   
                  downwind              111     3.35 [2.70, 4.27]   

                            high-confidence model  
eval_set          regime                           
all frames        overall       4.94 [4.53, 5.39]  
                  upwind        4.92 [4.46, 5.38]  
                  crosswind  18.75 [15.20, 22.31]  
                  downwind      5.89 [4.72, 9.45]  
confidence > 0.99 overall       4.22 [3.87, 4.63]  
                  upwind        4.06 [3.86, 4.54]  
                  crosswind  11.41 [10.26, 14.17]  
                  downwind      3.66 [2.85, 4.42]

Median per-trajectory error (deg) over the five seeds: mean ± sd (min–max)


full-data model  \
eval_set          regime                                  
all frames        overall       5.00 ± 0.05 (4.92–5.04)   
                  upwind        5.02 ± 0.09 (4.92–5.14)   
                  crosswind  19.79 ± 1.11 (17.84–20.65)   
                  downwind      5.26 ± 0.24 (5.04–5.64)   
confidence > 0.99 overall       4.34 ± 0.10 (4.21–4.46)   
                  upwind        4.26 ± 0.22 (3.98–4.59)   
                  crosswind  12.35 ± 0.52 (11.64–12.83)   
                  downwind      3.38 ± 0.21 (3.20–3.74)   

                                  high-confidence model  
eval_set          regime                                 
all frames        overall       5.28 ± 0.38 (4.94–5.88)  
                  upwind        5.22 ± 0.47 (4.92–6.05)  
                  crosswind  19.36 ± 1.55 (16.95–20.84)  
                  downwind      6.50 ± 0.59 (5.89–7.46)  
confidence > 0.99 overall       4.46 ± 0.43 (4.12–5.16)  
                  upwind        4.45 ± 0.53 (4.06–5.35)  
                  crosswind  12.27 ± 0.52 (11.41–12.69)  
                  downwind      3.57 ± 0.31 (3.22–3.89)

The comparison (per-trajectory paired differences, error against heading confidence, error distributions) is plotted in `07_full_vs_high_confidence_comparison.ipynb`, which reads `../tables/ann_conf0.99_test_frame_errors.csv`.